### Dell Laptop Price Analysis - PriceOye.pk

In [4]:
import time
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

### Data Collection Setup

In [5]:
options = Options()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()), options=options
)

url = "https://priceoye.pk/laptops/pricelist?brands=dell"
driver.get(url)
time.sleep(5)
soup = BeautifulSoup(driver.page_source, "html.parser")
all_products = []
products = soup.find_all("div", class_="productBox")

for product in products:
    out_of_stock = product.find("div", id="stock-badge", class_="out-of-stock-label")
    if out_of_stock:
        continue

    name = product.find("div", class_="p-title")
    name = name.get_text(strip=True) if name else None

    curr_price = product.find("div", class_="price-box p1")
    curr_price = (
        curr_price.get_text(strip=True).replace("Rs", "").replace(",", "")
        if curr_price
        else None
    )

    orig_price = product.find("div", class_="price-diff-retail")
    orig_price = (
        orig_price.get_text(strip=True).replace("Rs", "").replace(",", "")
        if orig_price
        else None
    )

    disc_percent = product.find("div", class_="price-diff-saving")
    disc_percent = disc_percent.get_text(strip=True) if disc_percent else None

    try:
        curr_price_val = int(curr_price) if curr_price else None
        orig_price_val = int(orig_price) if orig_price else None
    except:
        curr_price_val = None
        orig_price_val = None

    discount_amount = None
    if curr_price_val and orig_price_val:
        discount_amount = orig_price_val - curr_price_val

    all_products.append(
        {
            "Name": name,
            "Current Price (Rs)": curr_price_val,
            "Original Price (Rs)": orig_price_val,
            "Discount Amount (Rs)": discount_amount,
            "Discount %": disc_percent,
        }
    )

driver.quit()

df = pd.DataFrame(all_products)
print(df.to_string(index=False))
df.to_csv("dell_laptop.csv", index=False, encoding="utf-8-sig")

print("\nData saved to dell_laptop.csv")

                                                                 Name  Current Price (Rs)  Original Price (Rs)  Discount Amount (Rs) Discount %
                Dell Latitude 3540 Ci5-1335U (8GB-256GB SSD) 15.6 FHD              169999               249999                 80000    32% OFF
                           Dell Vostro 3530 Ci3-1305U (8GB-512GB SSD)              111999               143000                 31001    22% OFF
                           Dell Vostro 15 3530 13th Gen Core i5 1334u              142999               180000                 37001    21% OFF
        Dell Inspiron 15 3530 13th Gen Core i7 1355U (16GB-512GB SSD)              189999               219999                 30000    14% OFF
Dell Latitude 3550 Raptor Lake 13th Gen Core i7 1355U (8GB-512GB SSD)              269999               304999                 35000    11% OFF
 Dell Latitude 15 5540 15.6 Inches 13th Gen Core i7 DOS (8GB - 512GB)              344999               415000                 70001    

### Web Scraping and Data Collection

The data is structured to include current prices, original prices, and calculated discount amounts.

In [6]:
df = pd.read_csv("dell_laptop.csv")
df["Current Price (Rs)"] = pd.to_numeric(df["Current Price (Rs)"], errors="coerce")
df["Original Price (Rs)"] = pd.to_numeric(df["Original Price (Rs)"], errors="coerce")
df["Discount %"] = (
    df["Discount %"].astype(str).str.replace("% OFF", "", regex=False).str.strip()
)
df["Discount %"] = pd.to_numeric(df["Discount %"], errors="coerce")

# Price Range Analysis
cheapest = df.loc[df["Current Price (Rs)"].idxmin()]
expensive = df.loc[df["Current Price (Rs)"].idxmax()]
print(">>> Cheapest Laptop:")
print(cheapest[["Name", "Current Price (Rs)"]], "\n")
print(">>> Most Expensive Laptop:")
print(expensive[["Name", "Current Price (Rs)"]], "\n")

# Average Price Estimation
avg_price = df["Current Price (Rs)"].mean()
print(f">>> Average Dell Laptop Price: Rs {avg_price:,.0f}\n")

# Discount Insights
top_discounts = df.sort_values("Discount %", ascending=False).head(3)
print(">>> Top 3 Laptops with Highest Discounts:")
print(
    top_discounts[["Name", "Current Price (Rs)", "Original Price (Rs)", "Discount %"]],
    "\n",
)

# Model Comparison (Entry level)
if "Category" in df.columns:
    df["Category"] = df["Category"].astype(str).str.strip().str.title()
else:
    df["Category"] = "Entry-Level"
    df.loc[df["Name"].str.contains("i7|Ultra", case=False, na=False), "Category"] = (
        "High-End"
    )

avg_discounts = df.groupby("Category")["Discount %"].mean()

print(">>> Average Discounts by Model Category:")
for category, discount in avg_discounts.items():
    print(f"{category}: {discount:.2f}%")

if "High-End" in avg_discounts and "Entry-Level" in avg_discounts:
    if avg_discounts["High-End"] > avg_discounts["Entry-Level"]:
        print("\nHigh-End models generally get larger discounts.\n")
    else:
        print("\nHigh-End models generally get smaller discounts.\n")

# Customer Savings Potential
df["Savings"] = df["Original Price (Rs)"] - df["Current Price (Rs)"]
total_savings = df["Savings"].sum()

print(
    f">>> Total Savings if customer buys all {len(df)} laptops: Rs {total_savings:,.0f}\n"
)

>>> Cheapest Laptop:
Name                  DELL Vostro 15 3520 Laptop 12th Gen Core i3
Current Price (Rs)                                         106599
Name: 16, dtype: object 

>>> Most Expensive Laptop:
Name                  DELL LATTITUDE 5550 14TH GEN Core Ultra 7 155U...
Current Price (Rs)                                               360999
Name: 8, dtype: object 

>>> Average Dell Laptop Price: Rs 224,479

>>> Top 3 Laptops with Highest Discounts:
                                                 Name  Current Price (Rs)  \
0   Dell Latitude 3540 Ci5-1335U (8GB-256GB SSD) 1...              169999   
16        DELL Vostro 15 3520 Laptop 12th Gen Core i3              106599   
15       Dell Inspiron 15 3530 13th Gen Core i5 1334u              143999   

    Original Price (Rs)  Discount %  
0                249999          32  
16               149999          29  
15               200000          28   

>>> Average Discounts by Model Category:
Entry-Level: 20.64%
High-End: 12.33%

### Data Analysis and Insights

This section provides comprehensive analysis of the collected data:

1. Price Range Analysis:
   - Identifies the cheapest and most expensive Dell laptops
   - Calculates average laptop prices

2. Discount Analysis:
   - Finds top 3 laptops with highest discounts
   - Compares discounts across different categories

3. Market Segmentation:
   - Categorizes laptops into Entry-Level and High-End
   - Analyzes discount patterns across categories

4. Value Analysis:
   - Calculates potential customer savings
   - Provides insights into pricing strategies